# RQ5: Which EV brands deliver the best performance-to-price ratio?

**Hypothesis:** Budget-segment brands offer higher value (range/$ and hp/$), while luxury brands trade efficiency for premium features.

**Methodology:**
1. Compute composite performance score = (range_miles + horsepower) / price_usd * 1000
2. Rank brands by mean composite score
3. ANOVA across brands
4. Horizontal bar chart ranked by value score (PDF)
5. Brand comparison table (CSV)

In [ ]:
import pandas as pd, numpy as np, os, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import f_oneway
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family':'serif','font.size':10,'axes.titlesize':12,'axes.labelsize':11,'figure.dpi':300,'axes.spines.top':False,'axes.spines.right':False})

for p in ['/kaggle/input/electric-vehicle-market-and-pricing-dataset-2026/ev_market_2026.csv','ev_market_2026.csv']:
    if os.path.exists(p): df = pd.read_csv(p); break

df = df.dropna(subset=['brand','price_usd','range_miles','horsepower'])
df['value_score'] = (df['range_miles'] + df['horsepower']) / df['price_usd'] * 1000
print('Shape:', df.shape)
print('Brands:', df['brand'].nunique())

In [ ]:
brand_agg = df.groupby('brand').agg(
    N=('value_score','count'),
    Mean_Value=('value_score','mean'),
    SE_Value=('value_score', lambda x: x.std()/np.sqrt(len(x))),
    Mean_Price=('price_usd','mean'),
    Mean_Range=('range_miles','mean'),
    Mean_HP=('horsepower','mean'),
    Mean_Sales=('annual_sales_units','mean'),
).reset_index().sort_values('Mean_Value', ascending=False)

brands = brand_agg['brand'].tolist()
groups = [df[df['brand']==b]['value_score'].values for b in brands]
F, p_anova = f_oneway(*groups)
print(f'Brand ANOVA on value score: F={F:.2f}, p={p_anova:.4f}')
print(brand_agg[['brand','N','Mean_Value','Mean_Price','Mean_Range','Mean_HP']].round(2).to_string(index=False))

In [ ]:
# Colourmap: darker = lower price
import matplotlib.cm as cm
norm = plt.Normalize(brand_agg['Mean_Price'].min(), brand_agg['Mean_Price'].max())
colors = cm.RdYlGn_r(norm(brand_agg['Mean_Price'].values))

fig, ax = plt.subplots(figsize=(9, 7))
y = np.arange(len(brand_agg))
bars = ax.barh(y, brand_agg['Mean_Value'], xerr=brand_agg['SE_Value'],
               color=colors, capsize=4, error_kw={'linewidth':1.0,'ecolor':'#555'}, height=0.65)
ax.set_yticks(y)
ax.set_yticklabels(brand_agg['brand'], fontsize=9.5)
ax.set_xlabel('Value Score [(Range + HP) / Price × 1,000]')
ax.set_title(f'EV Brand Performance-to-Price Ratio\n(One-way ANOVA: F={F:.2f}, p={p_anova:.4f})', pad=12)
ax.axvline(brand_agg['Mean_Value'].mean(), color='#333', linewidth=1.2, linestyle='--', alpha=0.6, label='Grand mean')
ax.legend(fontsize=9, frameon=False)

# Colorbar for price
sm = cm.ScalarMappable(cmap='RdYlGn_r', norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.01)
cbar.set_label('Mean Price (USD)', fontsize=9)

plt.tight_layout()
fig.savefig('RQ5_Brand_Value_Score.pdf', bbox_inches='tight', format='pdf')
plt.show(); print('Saved: RQ5_Brand_Value_Score.pdf')

In [ ]:
out = brand_agg.copy()
out.columns = ['Brand','N','Mean Value Score','SE Value Score','Mean Price ($)','Mean Range (mi)','Mean HP','Mean Annual Sales']
out = out.round(3)
out['Mean Price ($)'] = out['Mean Price ($)'].round(0).astype(int)
out['Mean Annual Sales'] = out['Mean Annual Sales'].round(0).astype(int)
out.to_csv('RQ5_Brand_Summary_Table.csv', index=False)
print('Saved: RQ5_Brand_Summary_Table.csv'); out